In [2]:
import yaml
import random
import numpy as np 
import pandas as pd
from datetime import date, timedelta
from faker import Faker
from schema import(
    ProductSchema, SupplierSchema, NodeSchema,
    ScannerSchema, HubHierarchySchema
)
from coordinates import get_coordinates, zone_coordinates

fake = Faker("en_IN")

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

random.seed(cfg['simulation']["random_seed"])
np.random.seed(cfg["simulation"]["random_seed"])

SIM_START = date(2025,12,1)
SIM_END = date(2026,6,30)

print("Ready. ")

Ready. 


#### **Generate dim_supplier**

In [3]:
SUPPLIER_CATALOG = [
    # (name, city, edi_format, categories)
    # --------DAIRY--------
    ("Amul Cooperative North", "Delhi", "API", "dairy"),
    ("Amul Cooperative West", "Mumbai", "API", "dairy,frozen"),
    ("Amul Cooperative South", "Bangalore", "SFTP_CSV", "dairy"),
    ("Mother Dairy", "Delhi", "API", "dairy"),
    ("Nandini Dairy", "Bangalore", "SFTP_CSV", "dairy"),
    ("Fresh Farms Local", "Mohali", "email_manual", "dairy"),

    # --------Beverages--------
    ("Bisleri International", "Mumbai", "SFTP_CSV", "beverages"),
    ("Coca-Cola India", "Pune", "API", "beverages"),
    ("PepsiCo India", "Hyderabad", "API", "beverages"),
    ("Sunrise Distributors", "Indore", "email_manual", "beverages,snacks"),

    # --------STAPLES--------
    ("Fortune Foods", "Ahmedabad", "SFTP_CSV", "staples"),
    ("ITC Agri Business", "Hyderabad", "SFTP_CSV", "staples,snacks"),
    ("Ramesh Traders", "Surat", "email_manual", "staples,snacks"),

    # --------SNACKS--------
    ("Haldiram Snacks", "Nagpur", "SFTP_CSV", "snacks"),

    # --------Personal CARE AND HOUSEHOLD --------
    ("Hindustan Unilever", "Mumbai", "SFTP_CSV", "personal_care,household"),
    ("Colgate Palmolive", "Mumbai", "SFTP_XML", "personal_care,household"),
    ("Himalaya Wellness", "Bangalore", "SFTP_XML", "personal_care"),

    # -------FROZEN--------
    ("McCain Foods", "Pune", "SFTP_XML", "frozen"),
    ("Godrej Tyson Foods", "Mumbai", "SFTP_XML", "frozen"),

]

# mother hub id is needed for serves_mother_hub_ids
# assigned geographically - suppliers serve hubs in their region
SUPPLIER_HUB_MAP = {

    "Delhi": "MH_DEL_01",
    "Mumbai": "MH_MUM_01",
    "Bangalore": "MH_BLR_01",
    "Hyderabad": "MH_HYD_01",
    "Pune": "MH_MUM_01,MH_PUN_01",
    "Ahmedabad": "MH_AMD_01",
    "Nagpur": "MH_MUM_01,MH_HYD_01",
    "Surat": "MH_AMD_01",
    "Mohali": "MH_DEL_01",
    "Indore": "MH_AMD_01,MH_HYD_01",

}
CATEGORY_SUPPLIER_MAP = {
    "dairy": ["SUP_001", "SUP_002", "SUP_003", "SUP_004", "SUP_005", "SUP_006"],
    "beverages": ["SUP_007", "SUP_008", "SUP_009", "SUP_010"],
    "staples": ["SUP_011","SUP_012","SUP_013"],
    "snacks": ["SUP_010","SUP_013","SUP_014"],
    "personal_care": ["SUP_015","SUP_016","SUP_017"],
    "household": ["SUP_015","SUP_016"],
    "frozen": ["SUP_002","SUP_018","SUP_019"],
}

edi_cfg = cfg["suppliers"]["edi_formats"]

def generate_suppliers():
    suppliers = []
    for i, (name, city, edi, categories) in enumerate(SUPPLIER_CATALOG, 1):
        score_range = edi_cfg[edi]['quality_score_range']
        score = round(random.uniform(score_range[0], score_range[1]), 3)

        ## onboarded 1-5 years before simulation start
        days_back = random.randint(365, 1825)
        onboarded = SIM_START - timedelta(days = days_back)

        raw = {
            "supplier_id": f"SUP_{i:03d}",
            "supplier_name": name,
            "city": city,
            "edi_format": edi,
            "data_quality_score": score,
            "onboarded_date": onboarded,
            "categories_supplied": categories,
            "serves_mother_hub_ids": SUPPLIER_HUB_MAP.get(city, "MH_MUM_01")
        }

        validated = SupplierSchema(**raw)
        suppliers.append(validated.model_dump())

    return pd.DataFrame(suppliers)

df_supplier = generate_suppliers()
print(f"Suppliers: {len(df_supplier)}")
df_supplier.head()

Suppliers: 19


,supplier_id,supplier_name,city,edi_format,data_quality_score,onboarded_date,categories_supplied,serves_mother_hub_ids
0,SUP_001,Amul Cooperative North,Delhi,API,0.940,2024-10-11,dairy,MH_DEL_01
1,SUP_002,Amul Cooperative West,Mumbai,API,0.954,2023-07-19,"dairy,frozen",MH_MUM_01
2,SUP_003,Amul Cooperative South,Bangalore,SFTP_CSV,0.695,2024-05-06,dairy,MH_BLR_01
3,SUP_004,Mother Dairy,Delhi,API,0.945,2021-11-11,dairy,MH_DEL_01
4,SUP_005,Nandini Dairy,Bangalore,SFTP_CSV,0.667,2022-07-21,dairy,MH_BLR_01


### **Generate dim_product**

In [ ]:
PRODUCT_CATALOG = {
    "dairy": {
        "items": [
            # (name, U0M, min_price, max_price, shelf_life, weight_g, pack_ml_g, [list of brands])
            ("Full Cream Milk 500ml", "units", 24, 28, 7, 520, 500,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Full Cream Milk 1L", "units", 44, 52, 7, 1050, 1000,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Toned Milk 500ml", "units", 20, 24, 7, 520, 500,
             ["Amul", "Mother Dairy", "Parag"]),
            ("Paneer 200g", "units", 75, 95, 10, 210, None,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Butter 100g", "units", 52, 58, 90, 110, None,
             ["Amul", "Mother Dairy", "Parag"]),
            ("Curd 400g", "units", 38, 48, 5, 420, None,
             ["Amul", "Mother Dairy", "Nandini"]),
            ("Ice cream tub 500g","units", 350, 440, 180, 500, 500,
             ["Amul", "Kwality Walls", "Havmor"]),
        ]
    },

    "beverages": {
        "items": [
            ("Packaged Water 1L", "units", 15, 20, 365, 1050, 1000,
             ["Bisleri", "Kinley", "Aquafina"]),
            ("Packaged Water 500ml", "units", 10, 14, 365, 530, 500,
             ["Bisleri", "Kinley", "Aquafina"]),
            ("Cola 500ml", "units", 30, 40, 180, 540, 500,
             ["Pepsi", "Coca-Cola", "Thums Up"]),
            ("Fruit Juice 200ml", "units", 18, 28, 120, 220, 200,
             ["Paper Boat", "Real", "Tropicana"]),
            ("Energy Drink 250ml", "units", 85, 110, 180, 275, 250,
             ["Red Bull", "Monster"]),
        ]
    },

    "staples": {
        "items": [
            ("Basmati Rice 1kg", "kg", 85, 110, 365, 1020, None,
             ["India Gate", "Fortune", "Daawat"]),
            ("Toor Dal 500g", "units", 58, 78, 365, 520, None,
             ["Tata Sampann", "Fortune", "Aashirvaad"]),
            ("Atta 1kg", "kg", 48, 62, 180, 1020, None,
            ["Aashirvaad", "Pillsbury", "Fortune"] ),
            ("Sunflower Oil 1L", "units", 120, 155, 365, 1050, 1000,
             ["Fortune", "Saffola", "Sundrop"]),
            ("Sugar 1kg", "kg", 42, 52, 730, 1020, None,
             ["Uttam", "Dhampur", "Fortune"]),
        ]
    },

    "snacks": {
        "items": [
            ("Namkeen Mix 200g", "units", 38, 58, 180, 205, None,
             ["Haldiram", "Bikaji", "Balaji"]),
            ("Potato Chips 100g", "units", 18, 28, 90, 105, None,
             ["Lays", "Bingo", "Too Yum"]),
            ("Roasted Peanuts 150g", "units", 28, 42, 180, 158, None,
             ["Haldiram", "Bikaji", "Jabsons"]),
            ("Biscuits 200g", "units", 22, 35, 180, 210, None,
             ["Parle G", "Brittania", "Sunfeast"]),
            ("Instant Noodles 70g", "units", 12, 20, 180, 75, None, 
             ["Maggi", "Yipee", "Wai Wai"])
        ]
    },

    "household": {
        "items": [
            ("Dishwash Liquid 500ml", "units", 78, 120, 730, 530, 500,
             ["Vim", "Prill", "Exo"]),
            ("Floor Cleaner 1L", "units", 88, 140, 730, 1060, 1000,
             ["Lizol", "Colin", "Domex"]),
            ("Detergent Powder 1kg", "kg", 88, 130, 730, 1020, None,
             ["Surf Excel", "Ariel", "Tide"]),
            ("Toilet Cleaner 500ml", "units", 68, 98, 730, 540, 500, 
             ["Harpic", "Domex", "Sanifresh"]),
        ]
    },

    "personal_care": {
        "items": [
            ("Shampoo 200ml", "units", 120, 220, 730, 215, 200,
             ["Dove", "Head and Shoulders", "Pantene"]),
            ("Hand Wash 250ml", "units", 65, 110, 730, 265, 250,
             ["Dettol", "Lifebuoy", "Savlon"]),
            ("Toothpaste 100g", "units", 55, 95, 730, 108, None,
             ["Colgate", "Pepsodent", "Sensodyne"]),
            ("Face Wash 100ml", "units", 85, 160, 730, 108, 100,
             ["Himalaya", "Cetaphil", "Nivea"]),
            ("Moisturizer 50ml", "units", 120, 200, 730, 54, 50,
             ["Nivea", "Vaseline", "Dove"]),
        ]
    },

    "frozen": {
        "items": [
            ("Frozen Peas 500g", "units", 58, 88, 180, 520, None,
             ["McCain", "Mother Dairy", "ITC"]),
            ("Chicken Nuggets 400g", "units", 175, 245, 90, 420, None,
             ["Venky's", "Godrej Yummiez", "McCain"]),
            ("Veg Burger Patty 300g", "units", 95, 135, 90, 315, None,
             ["McCain", "ITC", "Amul"]),
            ("French Fries 400g", "units", 75, 120, 200,420 ,None,
             ["McCain", "Haldiram", "ITC"])
        ]
    },
}

# map category to supplier_id
CATEGORY_SUPPLIER_MAP = {
    "dairy": ["SUP_001", "SUP_002", "SUP_003", "SUP_004", "SUP_005", "SUP_006"],
    "beverages": ["SUP_007", "SUP_008", "SUP_009", "SUP_010"],
    "staples": ["SUP_011","SUP_012","SUP_013"],
    "snacks": ["SUP_010","SUP_013","SUP_014"],
    "personal_care": ["SUP_015","SUP_016","SUP_017"],
    "household": ["SUP_015","SUP_016"],
    "frozen": ["SUP_002","SUP_018","SUP_019"],
}

prod_cfg = cfg["product_categories"]

def generate_products():
    products = []
    prod_num = 1
    
    for category, data in PRODUCT_CATALOG.items():
        cat_cfg = prod_cfg[category]
        brands = data['brands']
        items = data["items"]
        suppliers = CATEGORY_SUPPLIER_MAP[category]

        for item in items:
            name, uom, min_p, max_p, shelf, weight, pack, item_brands = item
            for brand in item_brands[:3]: #top 3 brands per item
                if prod_num > cfg["products"]["total"]:
                    break

                price = round(random.uniform(min_p, max_p), 2)
                reorder = random.randint(20, 80)

                raw = {
                    "product_id": f"PROD_{prod_num:03d}",
                    "name": f"{brand} {name}",
                    "category": category,
                    "brand": brand,
                    "unit_of_measure": uom,
                    "base_price": price,
                    "pack_size_ml_or_g": pack,
                    "weight_grams": weight,
                    "shelf_life_days": shelf,
                    "storage_zone": cat_cfg["storage_zone"],
                    "reorder_point_units": reorder,
                    "supplier_id": random.choice(suppliers),
                }

                validated = ProductSchema(**raw)
                products.append(validated.model_dump())
                prod_num +=1

            if prod_num > cfg["products"]["total"]:
                break
        if prod_num > cfg["products"]["total"]:
            break

    return pd.DataFrame(products)

df_product = generate_products()
print(f"Products: {len(df_product)}")
df_product.head(10)   

### **Generate dim_node (Mother_hubs)**

In [ ]:
def generate_mother_hubs():
    nodes = []
    mh_cfg = cfg["mother_hubs"]

    for city, loc in mh_cfg["locations"].items():
        city_code = loc["city_code"]
        zone = loc["zone"]
        zone_code = loc["zone_code"]
        node_id = f"MH_{city_code}_01"

        total = random.randint(
            mh_cfg["capacity_total_units"][0],
            mh_cfg["capacity_total_units"][1],
        )

        cold_frac = random.uniform(
        mh_cfg["cold_fraction"][0],
        mh_cfg["cold_fraction"][1]
        )

        bev_frac = random.uniform(
            mh_cfg["beverage_fraction"][0],
            mh_cfg["beverage_fraction"][1]
        )

        if cold_frac + bev_frac > 0.85:
            bev_frac = 0.85-cold_frac

        cold = int(total* cold_frac)
        bev = int(total * bev_frac)
        dry = total - cold - bev

        lat, lon = get_coordinates(zone, city, "mother_hub", jitter = False)

        # opened 3-7 years before simulation start
        op_since = SIM_START - timedelta(days= random.randint(1095, 2555))

        raw = {
            "node_id": node_id,
            "node_type": "mother_hub",
            "node_name": f"{city} Mother Hub",
            "city": city,
            "zone": zone,
            "zone_type": "warehouse",
            "tier": "metropolitan",
            "parent_node_id": None,
            "capacity_units": total,
            "capacity_cold_units": cold,
            "capacity_beverage_units": bev,
            "capacity_dry_units": dry,
            "latitude": latitude,
            "longitude": longitude,
            "timezone": "Asia/Kolkata",
            "operational_since": op_since,
        }

        validated = NodeSchema(**raw)
        nodes.append(validated.model_dump())

    return nodes

mother_hub_rows = generate_mother_hubs()
print(f"Mother hubs: {len(mother_hub_rows)}")